# Preprocessing

In [1]:
import sys; sys.path.append("..")
import numpy as np
import pandas as pd
from pathlib import Path
from src.aggregate import aggregate
from src.moments import group_kurt

RAW = Path("../data/raw")
INTERIM = Path("../data/interim")

In [2]:
bb = pd.read_csv(RAW / "bureau_balance.csv")
bb["dpd"] = bb["STATUS"].map({"0": 0., "1": 1., "2": 2., "3": 3., "4": 4., "5": 5., "C": 0., "X": np.nan}).astype("float32")
bb["is_del"] = (bb["dpd"] > 0).astype("int8")
bb.shape

(27299925, 5)

# Feature Engineering

## Iteration 1

In [3]:
bb_agg = aggregate(bb, "SK_ID_BUREAU", "bb", cat_cols=["STATUS"])
bb_agg.shape

(817395, 33)

In [4]:
parts = []
for m in [3, 5, 6, 12, 15, 24, 60]:
    sub = bb[bb["MONTHS_BALANCE"] >= -m]
    w = sub.groupby("SK_ID_BUREAU")
    parts.append(w["dpd"].max().rename(f"bb_dpd_max_{m}"))
    parts.append(w["dpd"].mean().rename(f"bb_dpd_mean_{m}"))
    parts.append(w["dpd"].skew().rename(f"bb_dpd_skew_{m}"))
    parts.append(group_kurt(sub["dpd"], sub["SK_ID_BUREAU"]).rename(f"bb_dpd_kurt_{m}"))
    parts.append(w["is_del"].mean().rename(f"bb_del_share_{m}"))
windows = pd.concat(parts, axis=1)

frac = {}
for s, l in [(6, 12), (6, 24), (12, 24), (12, 60)]:
    frac[f"bb_dpd_mean_frac_{s}_{l}"] = windows[f"bb_dpd_mean_{s}"] / windows[f"bb_dpd_mean_{l}"]
    frac[f"bb_del_share_frac_{s}_{l}"] = windows[f"bb_del_share_{s}"] / windows[f"bb_del_share_{l}"]
frac = pd.DataFrame(frac).replace([np.inf, -np.inf], np.nan)

d = bb.dropna(subset=["dpd"])
g = d.groupby("SK_ID_BUREAU")
dx = d["MONTHS_BALANCE"] - g["MONTHS_BALANCE"].transform("mean")
dy = d["dpd"] - g["dpd"].transform("mean")
trend = (dx * dy).groupby(d["SK_ID_BUREAU"]).sum() / (dx * dx).groupby(d["SK_ID_BUREAU"]).sum()

bb_agg = bb_agg.join(windows).join(frac).join(trend.replace([np.inf, -np.inf], np.nan).rename("bb_dpd_trend"))
bb_agg.shape

(817395, 77)

## Iteration 2

In [5]:
recent_del = bb[bb["is_del"] == 1].groupby("SK_ID_BUREAU")["MONTHS_BALANCE"].max()
history_start = bb.groupby("SK_ID_BUREAU")["MONTHS_BALANCE"].min()
bb_agg["bb_months_since_last_del"] = -recent_del
bb_agg["bb_del_recency_ratio"] = (-recent_del) / (-history_start)
bb_agg = bb_agg.replace([np.inf, -np.inf], np.nan)
bb_agg.shape

(817395, 79)

# Save

In [6]:
bb_agg.reset_index().to_pickle(INTERIM / "bureau_balance.pkl")
bb_agg.shape

(817395, 79)